# Kolmogorov Flow (2D Incompressible, Periodic)

This notebook runs a **Kolmogorov flow** simulator: a 2D incompressible flow on a periodic domain driven by a sinusoidal body force.

## PDE

We evolve the **vorticity** \(\omega(x,y,t)\) and recover velocity from the streamfunction \(\psi\):

$$
\partial_t \omega + \mathbf{u}\cdot\nabla\omega
+  = \nu\nabla^2\omega + f_\omega(y) - \alpha\,\omega,
+$$

$$
\nabla^2\psi = -\omega,
+\qquad
+u = \partial_y\psi,
+\qquad
+v = -\partial_x\psi.
+$$

### Physics

- Incompressible 2D flow with viscosity \(\nu\) and optional linear drag \(\alpha\).
- A sinusoidal forcing maintains sustained dynamics and creates a canonical transition-to-turbulence testbed.

### Symbols

- \(\omega\): vorticity
- \(\psi\): streamfunction
- \(u,v\): velocity components
- \(\nu\): viscosity
- \(\alpha\): linear drag
- \(k_f\): forcing wavenumber

### Initial conditions

We start from low-pass filtered random vorticity (to avoid grid-scale noise dominating early dynamics).

### Boundary conditions

Periodic in both directions (implied by FFT-based spatial operators).

### Assumptions

- 2D, incompressible, periodic box.
- Pseudo-spectral spatial derivatives.
- Time integration via `scipy.integrate.solve_ivp` (RK45) on the method-of-lines system.

### What distinguishes this PDE

- Forced, incompressible dynamics with a simple, interpretable forcing structure.
- Useful as a benchmark for learning **advection-dominated** PDE dynamics and coherent structures.


In [ ]:
from IPython.display import HTML

from autosim.simulations import KolmogorovFlow
from autosim.utils import plot_spatiotemporal_video


In [ ]:
sim = KolmogorovFlow(return_timeseries=True, log_level="warning", n=64, T=0.2, dt=0.1)
batch = sim.forward_samples_spatiotemporal(n=1, random_seed=0, ensure_exact_n=True)
print(batch["data"].shape)

In [ ]:
# Use fixed, conservative parameters for interactive stability.
sim = KolmogorovFlow(
    return_timeseries=True,
    log_level="warning",
    n=32,       # faster grid
    L=2 * 3.141592653589793,
    T=10.0,     # still long enough to see structure
    dt=0.2,     # fewer saved frames
    kf=4,
    parameters_range={
        "nu": (0.01, 0.01),
        "forcing": (0.3, 0.3),
        "alpha": (0.2, 0.2),
    },
)
batch = sim.forward_samples_spatiotemporal(n=1, random_seed=0, ensure_exact_n=True)
print("data shape:", batch["data"].shape, "[batch, time, x, y, channels]")
print("channels:", sim.output_names)
print("params:", batch["constant_scalars"])

In [ ]:
anim = plot_spatiotemporal_video(
    batch["data"],
    batch_idx=0,
    channel_names=sim.output_names,
    preserve_aspect=True,
)
HTML(anim.to_jshtml())
